# Classwork: The Analyst's Toolkit

**Names:** Emily Sanchez

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [1]:
import pandas as pd
import numpy as np 
import scipy.stats as stats
import statsmodels.api as sm

df = pd.read_csv("./customers.csv")
df.info()
df.head(26)


<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  40 non-null     int64  
 1   age          40 non-null     int64  
 2   income       38 non-null     float64
 3   spending     40 non-null     float64
 4   region       40 non-null     str    
 5   subscribed   40 non-null     str    
dtypes: float64(2), int64(2), str(2)
memory usage: 2.0 KB


,customer_id,age,income,spending,region,subscribed
0,1,25,61.3,39.1,West,No
1,2,25,54.9,42.1,South,Yes
2,3,51,105.1,59.9,South,No
3,4,39,NaN,49.9,South,No
4,5,43,72.4,48.3,East,No
5,6,44,84.7,47.1,East,No
6,7,48,92.9,60.0,East,Yes
7,8,21,53.3,36.0,South,No
8,9,39,71.9,48.9,East,Yes
9,10,25,70.4,42.1,east,Yes


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [2]:

# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?)
    #Income has missing values 

# HINT 2: one category label is written in the wrong case -> standardize it
    #Fix region so they're all capitliazed 

# HINT 3: one value is impossible for its column -> treat it as missing, then fill
    #income value is negative, fix  with abs

# tip: check .isna().sum() and the unique values of the category column

In [3]:
df.isna().sum()

customer_id    0
age            0
income         2
spending       0
region         0
subscribed     0
dtype: int64

In [4]:
#fix region so that its capitlaized 

df["region"].value_counts()
 
region_map = {
    "east":"East", 
    "west":"West",
    "south": "South", 
    "north":"North"
}


df["region"] = df["region"].replace(region_map)

df.head(20)


,customer_id,age,income,spending,region,subscribed
0,1,25,61.3,39.1,West,No
1,2,25,54.9,42.1,South,Yes
2,3,51,105.1,59.9,South,No
3,4,39,NaN,49.9,South,No
4,5,43,72.4,48.3,East,No
5,6,44,84.7,47.1,East,No
6,7,48,92.9,60.0,East,Yes
7,8,21,53.3,36.0,South,No
8,9,39,71.9,48.9,East,Yes
9,10,25,70.4,42.1,East,Yes


In [5]:
#income as absolute value 
df["income"] = df["income"].abs()

#replace nan with mean 
df['income'] = df['income'].fillna(df["income"].mean())

df.head(28)

,customer_id,age,income,spending,region,subscribed
0,1,25,61.300000,39.1,West,No
1,2,25,54.900000,42.1,South,Yes
2,3,51,105.100000,59.9,South,No
3,4,39,85.639474,49.9,South,No
4,5,43,72.400000,48.3,East,No
5,6,44,84.700000,47.1,East,No
6,7,48,92.900000,60.0,East,Yes
7,8,21,53.300000,36.0,South,No
8,9,39,71.900000,48.9,East,Yes
9,10,25,70.400000,42.1,East,Yes


**Explain each issue:** what you found and how you handled it.

For the first hint, I noticed that the incme had several missing values- NaN. This went hand in hand with hint 3, where a value was impossible. In this case, an income value was negative. I dealt with these together. First, I made each income value an absolute value- this took care of the negative accounts. Then, I took the mean of income and replaced all the NaN with the mean. I chose mean because mean is the better representation of the data and can give us a good idea of the data. We needed to find the abs() first so that the negative values did not bring the mean down. For Hint 2, I found that the region were sometimes capitalized and other times not. I decided to capitalize all the regions, so I mapped out the valid regions and fixed the non-standard ones. 



## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [6]:
# data types: income dataframe, spending dataframe 
# tool: pearson correlation using stats.pearsonr

print('Income & Spending')
r, p = stats.pearsonr(df['income'], df['spending'])
print(f'r (Pearson correlation coefficient): {r:.3f}')
print(f'p-value: {p:.2e}')

Income & Spending
r (Pearson correlation coefficient): 0.737
p-value: 5.84e-08


*Your answer:* Using correlation, we were able to find that there is a strong, positive correlation between customer income (income dataframe) and customer spending (spending dataframe), meaning there is reason to believe that there is a relationship between income and spending. 





### Q2. Is a customer's region related to whether they subscribed?

In [7]:
# data types: region = ?, subscribed = ?
# tool: ?
# hint: two categorical variables -> build a table with pd.crosstab, then test it

contingency_table = pd.crosstab(df['region'], df['subscribed'])
print(contingency_table)


subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1


*Your answer:* Using chi-square test and the contigency table, we are able to conclude that a cusotmer's region does relate to whether they are subscribed. 



### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [12]:
# data types: predicting a continuous outcome from TWO predictors
# tool: ?
# hint: remember to add a constant; look at R-squared and each predictor's p-value

x = df[['income', 'age']]
y = df['spending']

x = sm.add_constant(x)

model = sm.OLS(y, x).fit()     
predictions = model.predict(x)

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.557
Model:                            OLS   Adj. R-squared:                  0.533
Method:                 Least Squares   F-statistic:                     23.25
Date:                Mon, 05 Oct 2026   Prob (F-statistic):           2.89e-07
Time:                        16:18:48   Log-Likelihood:                -129.51
No. Observations:                  40   AIC:                             265.0
Df Residuals:                      37   BIC:                             270.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         20.0731      5.248      3.825      0.0

*Your answer:* The r value is 0.557, meaning about 55.7% of the data can be predicted, so income and age can predict most of the data 


### Q4. Which region has the most customers, and what is the average income overall?

In [9]:
# hint: count a categorical column
count_region = df["region"].value_counts()
print(count_region)


# take the mean of a numeric column
income_mean = df["income"].mean()
print (f"Average Income: {income_mean:.2f}")

region
East     21
West     10
South     9
Name: count, dtype: int64
Average Income: 85.64


*Your answer:* The Eastern region has the most customers and the average overall income is 85.64 thousand dollars, information we glean using descriptive statistics



## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:* The hints in each section were really helpful in understanding what tool to use. The information we needed for the analysis depended on the tool, so understanding what each tool does helped. 


**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:* No we cannot say this because correlation does not equal causation


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [15]:
# hint: chi2_contingency returns the expected table as its 4th value


chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table, correction=False) 

expected_df = pd.DataFrame(expected, index = contingency_table.index, columns = contingency_table.columns)
print(expected_df.round(2))

# check: are all expected counts >= 5?

if (expected >=5).all():
  print("Assumption Okay; all counts greater than 5.")
else:
  print("WARNING! some expected counts may be less than 5.")


subscribed    No   Yes
region                
East        10.5  10.5
South        4.5   4.5
West         5.0   5.0
WARNING! some expected counts may be less than 5.


*Your answer:* i would tell the compnay ot be wary of the numbers here- the data can be slightly trusted, but taken with a grain of salt. It is likley that the region does not have to do with if they are subscribed or not. 


### Push your code to GitHub